# Qualitative figure (UNet++, Exp 1 vs Exp 3)

Set the two checkpoint paths and the frames/crops in the CONFIG cell, then run all cells. Output: `outputs/fig_qualitative.png`.


In [ ]:
!pip install --no-index --find-links /kaggle/input/datasets/hinlphmhng/uav-seg-offline-assets/offline_assets/packages segmentation-models-pytorch albumentations omegaconf torchmetrics

In [ ]:
# ============================================================
# CONFIG
# ============================================================
MODEL_NAME, ENCODER, SEED, USE_AMP = 'unetpp', 'resnet50', 123, True
ENCODER_WEIGHT_DIR = '/kaggle/input/datasets/itmeagain77/resnet50-pytorch-weights'

# best.pth of the two UNet++ seed-123 runs (add their outputs as input datasets)
CKPT_EXP1 = '/kaggle/input/<dataset-of-UNet++-Exp1-s123>/best.pth'      # old checkpoints also work
CKPT_EXP3 = '/kaggle/input/<dataset-of-R01_unetpp_adaptive_s123>/checkpoints/best.pth'

# (sequence, frame number in the file name, (top, left) of the crop)
FRAMES = [('seq6', 290, (192, 320)), ('seq11', 3701, (192, 320)), ('seq6', 654, (192, 320))]
CROP = (384, 640)            # crop height, width (full frame is 768 x 1280)
GAMMA = 0.7                  # display-only brightening of the input image


In [ ]:
import os
from pathlib import Path

SUNNY_DIRS = [
    Path('/kaggle/input/datasets/leehien12/uav-forest-sunny-part1'),
    Path('/kaggle/input/datasets/khesng/uav-forest-sunny-next'),
    Path('/kaggle/input/datasets/leehien12/uav-forest-sunny'),
]
OVERCAST_DIRS = [
    Path('/kaggle/input/datasets/hinlphmhng/uav-forest-overcast-1'),
    Path('/kaggle/input/datasets/hinlphmhng/uav-forest-overcast-2'),
]
SPLIT_CONFIG = {
    'train': ['seq1', 'seq3', 'seq4', 'seq5', 'seq8', 'seq9',
              'seq10', 'seq12', 'seq14', 'seq15', 'seq16', 'seq17'],
    'val':   ['seq2', 'seq13', 'seq18'],
    'test':  ['seq6', 'seq7', 'seq11'],
}

def collect_pairs_from_dirs(base_dirs, split_config):
    data = {'train': [], 'val': [], 'test': []}
    found = {}
    for base_path in base_dirs:
        if not Path(base_path).exists():
            print(f'  [WARN] missing: {base_path}')
            continue
        for color_dir in Path(base_path).rglob('color'):
            if not color_dir.is_dir():
                continue
            seq_dir, seq_name = color_dir.parent, color_dir.parent.name
            mask_dir = seq_dir / 'labels'
            if not mask_dir.exists():
                mask_dir = seq_dir / 'label'
            if not mask_dir.exists():
                continue
            target = next((k for k, v in split_config.items() if seq_name in v), None)
            if target is None:
                continue
            n = 0
            for img_p in sorted(list(color_dir.glob('*.png')) + list(color_dir.glob('*.jpg'))):
                mask_p = (mask_dir / img_p.name).with_suffix('.png')
                if not mask_p.exists():
                    mask_p = (mask_dir / img_p.name).with_suffix('.jpg')
                if mask_p.exists():
                    data[target].append((str(img_p), str(mask_p)))
                    n += 1
            found[seq_name] = (target, n)
    return data, found

data, found_seqs = collect_pairs_from_dirs(SUNNY_DIRS + OVERCAST_DIRS, SPLIT_CONFIG)
train_files, val_files, test_files = data['train'], data['val'], data['test']
missing = set(sum(SPLIT_CONFIG.values(), [])) - set(found_seqs)
assert not missing, f'Missing sequences: {sorted(missing)}'
print(f'TRAIN {len(train_files):,} | VAL {len(val_files):,} | TEST {len(test_files):,}')
assert (len(train_files), len(val_files), len(test_files)) == (17454, 4297, 4491), 'Unexpected split sizes'


In [ ]:
import numpy as np
from PIL import Image

LABEL_COLORS = [
    (0, 255, 255), (0, 127, 0), (19, 132, 69), (0, 53, 65), (130, 76, 0),
    (152, 251, 152), (151, 126, 171), (250, 150, 0), (115, 176, 195),
    (123, 123, 123), (0, 0, 0),
]
CLASS_NAMES = ['Sky', 'Deciduous_trees', 'Coniferous_trees', 'Fallen_trees',
               'Dirt_ground', 'Ground_vegetation', 'Rocks', 'Building',
               'Fence', 'Car', 'Empty']
NUM_CLASSES = len(CLASS_NAMES)   # 11, Empty = index 10
EMPTY_CLASS = NUM_CLASSES - 1
PALETTE = np.array(LABEL_COLORS, dtype=np.uint8)

RGB_TO_CLASS_LUT = np.full((256, 256, 256), EMPTY_CLASS, dtype=np.uint8)
for cid, color in enumerate(LABEL_COLORS):
    RGB_TO_CLASS_LUT[color] = cid

def fast_rgb_to_mask(rgb_arr):
    return RGB_TO_CLASS_LUT[rgb_arr[..., 0], rgb_arr[..., 1], rgb_arr[..., 2]]


In [ ]:
import random
import torch

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
AMP_DTYPE_T = torch.bfloat16 if globals().get('AMP_DTYPE', 'float16') == 'bfloat16' else torch.float16
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('[INFO] Device:', torch.cuda.get_device_name() if DEVICE == 'cuda' else 'cpu')

OUTPUT_DIR = Path('/kaggle/working/outputs') if os.path.exists('/kaggle') else Path('outputs')
CKPT_DIR = OUTPUT_DIR / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import shutil, glob
import torch.nn as nn
import segmentation_models_pytorch as smp

CACHE_DIR = '/root/.cache/torch/hub/checkpoints/'
os.makedirs(CACHE_DIR, exist_ok=True)
cache_name = {'resnet50': 'resnet50-19c8e357.pth', 'mit_b4': 'mit_b4.pth'}[ENCODER]
pth_files = glob.glob(os.path.join(ENCODER_WEIGHT_DIR, '*.pth'))
if pth_files and not os.path.exists(os.path.join(CACHE_DIR, cache_name)):
    shutil.copy(pth_files[0], os.path.join(CACHE_DIR, cache_name))
print('[OK] encoder weights:', pth_files[:1] or 'download from internet')

MODELS = {
    'unet':          smp.Unet,
    'unetpp':        smp.UnetPlusPlus,
    'deeplabv3plus': smp.DeepLabV3Plus,
    'segformer':     smp.Segformer,
    'manet':         smp.MAnet,
    'upernet':       smp.UPerNet,
}
model = MODELS[MODEL_NAME](encoder_name=ENCODER, encoder_weights='imagenet',
                           in_channels=3, classes=NUM_CLASSES).to(DEVICE)
print(f'[MODEL] {MODEL_NAME}/{ENCODER} | params {sum(p.numel() for p in model.parameters()):,}')


In [ ]:
import re
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import albumentations as A
from albumentations.pytorch import ToTensorV2

val_tf = A.Compose([A.Resize(height=768, width=1280),
                    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), ToTensorV2()])

def load_model(path):
    m = MODELS[MODEL_NAME](encoder_name=ENCODER, encoder_weights=None, in_channels=3, classes=NUM_CLASSES)
    ck = torch.load(path, map_location=DEVICE, weights_only=False)
    m.load_state_dict(ck['model_state_dict'])
    print(f'loaded {path} (epoch {ck.get("epoch")}, val mIoU {ck.get("miou")})')
    return m.to(DEVICE).eval()

def find_frame(seq, frame):
    cands = [p for p in test_files if Path(p[0]).parent.parent.name == seq]
    hit = [p for p in cands if re.findall(r'\d+', Path(p[0]).stem) and int(re.findall(r'\d+', Path(p[0]).stem)[-1]) == frame]
    if not hit:
        raise ValueError(f'{seq} frame {frame} not found; example names: {[Path(p[0]).name for p in cands[:3]]}')
    return hit[0]

@torch.no_grad()
def predict(m, img):
    x = val_tf(image=img)['image'].unsqueeze(0).to(DEVICE)
    with torch.amp.autocast('cuda', enabled=USE_AMP):
        return m(x).argmax(1)[0].cpu().numpy()

models = {'Exp 1 (CE)': load_model(CKPT_EXP1), 'Exp 3': load_model(CKPT_EXP3)}
ch, cw = CROP
fig, axes = plt.subplots(len(FRAMES), 4, figsize=(12, 2.0 * len(FRAMES) + 0.6))
axes = np.atleast_2d(axes)
for r, (seq, frame, (top, left)) in enumerate(FRAMES):
    img_p, mask_p = find_frame(seq, frame)
    img = np.array(Image.open(img_p).convert('RGB'))
    gt = fast_rgb_to_mask(np.array(Image.open(mask_p).convert('RGB')))
    panels = [np.clip(255 * (img / 255.0) ** GAMMA, 0, 255).astype(np.uint8), PALETTE[gt]]
    panels += [PALETTE[predict(m, img)] for m in models.values()]
    titles = ['Input', 'Ground truth'] + list(models)
    for c, (p, t) in enumerate(zip(panels, titles)):
        ax = axes[r, c]
        ax.imshow(p[top:top + ch, left:left + cw]); ax.set_xticks([]); ax.set_yticks([])
        if r == 0:
            ax.set_title(t, fontsize=11)
        if c == 0:
            ax.set_ylabel(f'{seq}, frame {frame}', fontsize=9)
legend = [Patch(facecolor=PALETTE[i] / 255.0, edgecolor='#999999', label=n.replace('_', ' '))
          for i, n in enumerate(CLASS_NAMES[:-1])]
fig.legend(handles=legend, loc='lower center', ncol=10, fontsize=8, frameon=False, bbox_to_anchor=(0.5, -0.01))
fig.tight_layout(rect=(0, 0.04, 1, 1))
fig.savefig(OUTPUT_DIR / 'fig_qualitative.png', dpi=300, bbox_inches='tight')
print('saved', OUTPUT_DIR / 'fig_qualitative.png')
